# *** Create WB Diff function ***

# 0. Initial

## 0.0. Imports

In [1]:
import flopy as fp
import pandas as pd
from WS_Mdl.core.mdl import Mdl_N
from WS_Mdl.core.style import set_verbose
import numpy as np

from WS_Mdl.imod.msw.mete_grid import to_DF
from WS_Mdl.imod.msw.meteo import to_XA
from WS_Mdl.imod.prj import r_with_OBS
from WS_Mdl.xr.spatial import clip_Mdl_area as xr_clip_Mdl_area
import xarray as xra

import imod
from WS_Mdl.core.defaults import CRS
from WS_Mdl.core.style import style_reset, blue
from WS_Mdl.imod.ini import CeCes
from WS_Mdl.xr import diagnostics
from WS_Mdl.imod.pop.msw import load_Par_Out

## 0.1 Options

In [2]:
set_verbose(False)
MdlN='NBr52'
MdlN_B='NBr51'
date='19961231'
cumulative=True
sum_Pkg=True
net_only = False
Pa_Out=None
x_CeCes, y_CeCes = CeCes(MdlN)
set_verbose(True)

# 1. MF6 budget

In [3]:
# Load basics
M = Mdl_N(MdlN)
M_B = Mdl_N(MdlN_B) if MdlN_B else Mdl_N(M.B)
dx = dy = float(M.INI.CELLSIZE)

In [4]:
start_date = pd.to_datetime( M.INI.SDATE, format='%Y%m%d')

In [5]:
# Load budget to dataframes. fp.utils.Mf6ListBudget returns a tuple. 1st item is WB for each SP. 2nd item is cumulative.
i = 1 if cumulative else 0
DF_1 = fp.utils.Mf6ListBudget(M.Pa.LST_Mdl).get_dataframes(start_datetime=start_date - pd.Timedelta(days=1), diff=net_only)[i]
DF_2 = fp.utils.Mf6ListBudget(M_B.Pa.LST_Mdl).get_dataframes(start_datetime=start_date - pd.Timedelta(days=1), diff=net_only)[i]

In [6]:
if date is None:
    date = min(DF_1.index[-1], DF_2.index[-1]).strftime('%Y-%m-%d') # Use latest common date if not specified

In [7]:
S_1 = DF_1.loc[DF_1.index == date].squeeze()
S_2 = DF_2.loc[DF_2.index == date].squeeze()

In [8]:
S_1.index = S_1.index.str.upper()
S_2.index = S_2.index.str.upper()

In [9]:
DF = pd.DataFrame([S_1, S_2], index=[MdlN, MdlN_B]).T

In [10]:
DF.drop(index=['TOTAL_IN', 'TOTAL_OUT', 'total_in', 'total_out', 'percent_discrepancy', 'in-out', 'total', 'PERCENT_DISCREPANCY', 'IN-OUT', 'TOTAL'], inplace=True, errors='ignore')

In [11]:
if sum_Pkg:
    S_idx = DF.index.to_series().astype(str)
    S_suffix = np.where(
        S_idx.str.endswith('_IN'),
        '_IN',
        np.where(S_idx.str.endswith('_OUT'), '_OUT', '_NET'),
    )
    S_grp = S_idx.str[:3] + S_suffix
    DF = DF.groupby(S_grp, sort=False).sum(min_count=1)

In [12]:
if not net_only:
    # Add net rows
    S_i = DF.index.to_series()
    S_i_in = S_i[S_i.str.endswith('_IN')]
    S_i_out = S_i[S_i.str.endswith('_OUT')]
    S_i_net = S_i_in.str.replace('_IN', '_NET')
    DF_NET = DF.loc[S_i_in.values].set_axis(S_i_net.values, axis=0) - DF.loc[S_i_out.values].to_numpy()
    DF = pd.concat([DF, DF_NET], axis=0)

In [13]:
# Sort index
sorted_i = pd.concat( [i.sort_values() for i in [S_i_in, S_i_out, S_i_net] ]).values
DF = DF.loc[sorted_i]

# 2. Load MSW budget

In [36]:
for m in [MdlN, MdlN_B]:

    # Load MSW WB Out CSV and handle dates
    DF_MSW = pd.read_csv(M.Pa.MSW / 'msw/csv/tot_svat_dtgw.csv', index_col=False, skipinitialspace=True)
    DF_MSW['TimeStamp'] = pd.to_datetime(DF_MSW['TimeStamp'].str.replace(' 24:', ' 00:', regex=False)) # Convert TimeStamp to datetime, handling '24:' as '00:'

    # Calculate Sum for the same period as GW budget and append to DF
    date_selection = (DF_MSW['TimeStamp']>=start_date) & (DF_MSW['TimeStamp']<=pd.to_datetime(date))
    DF.loc['P', MdlN] = float(DF_MSW.loc[date_selection, 'Pm(m3)'].sum())
    DF.loc['AET', MdlN] = float(DF_MSW.loc[date_selection, ['Esp(m3)', 'Eic(m3)', 'Epd(m3)', 'Ebs(m3)', 'Tact(m3)']].sum().sum())
    DF.loc['qrun', MdlN] = float(DF_MSW.loc[date_selection, 'qrun(m3)'].sum())

In [37]:
DF

,NBr52,NBr51
CHD_IN,9.143000e+06,1.231651e+07
DRN_IN,0.000000e+00,0.000000e+00
RCH_IN,1.533264e+07,2.953937e+07
RIV_IN,2.056253e+02,0.000000e+00
SFR_IN,NaN,1.927823e+05
STO_IN,1.577510e+07,2.089386e+07
WEL_IN,0.000000e+00,0.000000e+00
CHD_OUT,1.364053e+07,1.361714e+07
DRN_OUT,8.021603e+05,3.699198e+06
RCH_OUT,4.535562e+06,7.496466e+06


In [61]:
dec_cols = [i for i in DF_MSW.columns if 'decS' in i]

In [62]:
DF_MSW.loc[date_selection, dec_cols].iloc[:, 2:].groupby(DF_MSW['TimeStamp'].dt.year).sum()

,decSpdmic(m3),decS01(m3),decS02(m3),decS03(m3),decS04(m3),decS05(m3),decS06(m3),decS07(m3),decS08(m3),decS09(m3),decS10(m3),decS11(m3),decS12(m3),decS13(m3),decS14(m3),decS15(m3),decS16(m3),decS17(m3),decS18(m3)
TimeStamp,,,,,,,,,,,,,,,,,,,
1996,-10147.630697,-1.315412e+06,-395965.792123,2.711638e+06,455456.72135,18179.177884,274.567822,0.021722,-0.071044,-0.023365,0.000073,0.0,-0.000146,-0.000219,-0.000146,-0.008762,-0.010952,0.000146,0.04673


In [64]:
DF_MSW.loc[date_selection, dec_cols].iloc[:, 2:].groupby(DF_MSW['TimeStamp'].dt.year).sum().sum().sum()

np.float64(1464022.4826879818)

# 3. Test WB closing

In [45]:
DF_MSW.loc[date_selection, ['Psgw(m3)']].sum(), DF_MSW.loc[date_selection, ['Pssw(m3)']].sum()

(Psgw(m3)    2.749410e+06
 dtype: float64,
 Pssw(m3)    0.0
 dtype: float64)

In [43]:
DF.loc['P', MdlN] + DF.loc['AET', MdlN] - DF.loc['qrun', MdlN]

np.float32(2.0634036e+07)

# 4. Aggregate

In [ ]:

DF['Diff'] = DF[MdlN] - DF[MdlN_B]
DF = DF.replace([np.inf, -np.inf], np.nan).round(0).astype('Int64')
DF['Diff_%'] = DF.apply(
    lambda x: x['Diff'] / x[MdlN_B] * 100 if pd.notnull(x['Diff']) and x[MdlN_B] != 0 else np.nan, axis=1
)
# Replace infinities, convert to nullable Int64, and display missing values as '-'
DF = DF.replace([np.inf, -np.inf], np.nan).round(0).astype('Int64')
DF.style.format(na_rep='-')
if Pa_Out is None:
    Pa_Out = M.Pa.PoP_Out_MdlN / f'WB_Diff_{MdlN}_vs_{MdlN_B}_{date}.xlsx'

Pa_Out = Path(Pa_Out)
Pa_Out.parent.mkdir(parents=True, exist_ok=True)
DF.to_excel(Pa_Out, index=True, na_rep='-')
sprint(f'🟢🟢🟢 - Saved WB stage to {Pa_Out}.')


# -1. Junkyard - Old method of loading MSW IDF Outs (much slower than laoding the budget csv)

In [ ]:
# (DA_P*DA_P_A).sum(dim=['time', 'x', 'y']).values

array([6.1860744e+07], dtype=float32)

In [ ]:
# (DA_AEVT*DA_AEVT_A).sum(dim=['time', 'x', 'y']).values

array([-4.456649e+07], dtype=float32)

In [ ]:
# (DA_qrun*DA_qrun_A).sum(dim=['time', 'x', 'y']).values

array([-3.339785e+06], dtype=float32)

# 2. Load P

In [ ]:
# DF_ = DF.copy()

In [ ]:
# DF_P_prev = None
# for m in [MdlN, MdlN_B]:
#     PRJ = r_with_OBS(M.Pa.PRJ)[0] # [0], cause [1] is the OBS

#     DF_P = to_DF(PRJ)

#     if not cumulative:
#         DF_P = DF_P.loc[DF_P['DT'] == pd.to_datetime(date)]
#     else:
#         DF_P = DF_P.loc[(DF_P['DT'] <= pd.to_datetime(date )) & (DF_P['DT'] >= start_date)]

#     if DF_P_prev is not None:
#         if DF_P['P'].equals(DF_P_prev):
#             print(f' -- {blue}{m}{style_reset}: Copied P from previous MdlN (identical).')
#             continue
#     else:
#         DF_P_prev = DF_P['P'].copy()

#     A_P = to_XA(DF_P, 'P', m)
#     A_P_refined = A_P.interp(coords={'x': x_CeCes, 'y': y_CeCes[::-1]}, method='nearest') # Refines and clips

#     DF.loc['P', m] = P = A_P_refined.sum(dim=['time', 'x', 'y']).values # asign to DF

 -- NBr52: Loading P: 100%|██████████| 366/366 [00:14<00:00, 25.42it/s]


 -- NBr51: Copied P from previous MdlN (identical).


# 3. Load AEVT

In [ ]:
# start_date = start_date.strftime('%Y%m%d')

In [ ]:
# DA_AEVT, DA_AEVT_A = load_Par_Out(MdlN, 'bdgETact', start_date, date) 

In [ ]:
# DA_P, DA_P_A = load_Par_Out(MdlN, 'bdgPm', start_date, date) 

# 3. Load Surface runoff

In [ ]:
# DA_qrun, DA_qrun_A = load_Par_Out(MdlN, 'bdgqrun', start_date, date) 